# 📱 Viral Video Agent — Optional Official API Mode

Open in Google Colab on Android Chrome, then **Runtime → Run all**. YouTube provides popular-video metadata/links only; Pexels provides downloadable licensed MP4 clips. This mode needs at least one API key. Never share your keys or put them in GitHub.

Use the **No Key** notebook in the same GitHub folder if you have no API keys.

- YouTube Data API: https://console.cloud.google.com/
- Pexels API: https://www.pexels.com/api/
- Source terms: https://developers.google.com/youtube/terms/developer-policies and https://www.pexels.com/license/


In [ ]:
!pip -q install "requests>=2.32.0,<3" "python-dotenv>=1.0.1,<2"


In [ ]:
import os
from getpass import getpass

print('🔑 Enter your API keys (input will be hidden). At least one is needed.')
os.environ['YOUTUBE_API_KEY'] = getpass('YouTube Data API key (Enter to skip): ').strip()
os.environ['PEXELS_API_KEY'] = getpass('Pexels API key (Enter to skip): ').strip()
if not (os.environ['YOUTUBE_API_KEY'] or os.environ['PEXELS_API_KEY']):
    raise ValueError('You need a YouTube and/or Pexels API key; get the keys in the links above, then rerun this cell.')

os.environ['YOUTUBE_TOPICS'] = 'GTA 6,Marvel'
os.environ['YOUTUBE_REGION'] = 'IN'
os.environ['LOOKBACK_DAYS'] = '7'
os.environ['TOP_YOUTUBE'] = '10'
os.environ['PEXELS_MODE'] = 'search'
os.environ['PEXELS_QUERY'] = 'gaming'
os.environ['PEXELS_ORIENTATION'] = 'portrait'
os.environ['MAX_DOWNLOADS'] = '3'
os.environ['MAX_FILE_MB'] = '45'
print('✅ Configured. YouTube returns links; Pexels can return licensed MP4 files.')


### Agent source code
The cell below installs the agent locally inside this Colab session. It uses official API data and only downloads Pexels files from permitted API-provided links.

In [ ]:
%%writefile agent.py
"""Viral Video Finder - discovery and licensed Pexels downloads only.

YouTube: metadata/links only; do not download YouTube audiovisual content.
Pexels: retrieve and download video files from API-provided links under its license.
"""
from __future__ import annotations

import argparse
import html
import json
import os
import re
import sys
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

import requests
from dotenv import load_dotenv

load_dotenv()

YT_API = "https://www.googleapis.com/youtube/v3"
PEXELS_API = "https://api.pexels.com/v1/videos"
TG_API = "https://api.telegram.org"
OUTPUT = Path("output")
DOWNLOADS = OUTPUT / "downloads"
USER_AGENT = "ViralVideoFinder/1.0 (research-and-authorized-media-download)"


def env_int(key: str, default: int, minimum: int = 1, maximum: int = 100) -> int:
    try:
        return max(minimum, min(maximum, int(os.getenv(key, str(default)))))
    except ValueError:
        return default


def api_get(url: str, *, params: dict[str, Any] | None = None, headers: dict[str, str] | None = None) -> dict[str, Any]:
    r = requests.get(url, params=params, headers=headers, timeout=(10, 35))
    r.raise_for_status()
    return r.json()


def safe_error(exc: Exception) -> str:
    # Prevent request URLs (and query-string API keys / bot tokens) from leaking into logs.
    if isinstance(exc, requests.HTTPError) and exc.response is not None:
        return f"HTTP {exc.response.status_code}"
    return type(exc).__name__


def iso_duration_seconds(s: str) -> int:
    match = re.fullmatch(r"P(?:\d+D)?T(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", s or "")
    if not match:
        return 0
    h, m, sec = (int(v or 0) for v in match.groups())
    return h * 3600 + m * 60 + sec


def discover_youtube(api_key: str) -> list[dict[str, Any]]:
    if not api_key:
        print("[YouTube] API key missing: skipping.")
        return []
    topics = [x.strip() for x in os.getenv("YOUTUBE_TOPICS", "GTA 6,Marvel").split(",") if x.strip()][:3]
    region = os.getenv("YOUTUBE_REGION", "IN").strip().upper()
    days = env_int("LOOKBACK_DAYS", 7, 1, 30)
    published_after = (datetime.now(timezone.utc) - timedelta(days=days)).isoformat(timespec="seconds").replace("+00:00", "Z")
    found: dict[str, dict[str, Any]] = {}
    for topic in topics:
        try:
            response = api_get(f"{YT_API}/search", params={
                "key": api_key, "part": "snippet", "type": "video", "q": topic,
                "order": "viewCount", "publishedAfter": published_after,
                "maxResults": 20, "regionCode": region,
                "videoDuration": "short",  # YouTube definition: under 4 minutes
            })
            for entry in response.get("items", []):
                vid = entry.get("id", {}).get("videoId")
                if vid:
                    found[vid] = {"id": vid, "topic": topic}
        except (requests.RequestException, ValueError) as e:
            print(f"[YouTube] Search error for {topic}: {safe_error(e)}")
    if not found:
        return []
    video_items: list[dict[str, Any]] = []
    ids = list(found.keys())
    for start in range(0, len(ids), 50):
        try:
            data = api_get(f"{YT_API}/videos", params={
                "key": api_key, "part": "snippet,statistics,contentDetails",
                "id": ",".join(ids[start:start + 50]),
            })
            video_items.extend(data.get("items", []))
        except (requests.RequestException, ValueError) as e:
            print(f"[YouTube] Statistics error: {safe_error(e)}")
    now = datetime.now(timezone.utc)
    results = []
    for video in video_items:
        video_id = video.get("id", "")
        details = video.get("contentDetails", {})
        seconds = iso_duration_seconds(details.get("duration", ""))
        if seconds <= 0 or seconds > 180:
            continue
        snippet = video.get("snippet", {})
        stats = video.get("statistics", {})
        try:
            published = datetime.fromisoformat(snippet.get("publishedAt", "").replace("Z", "+00:00"))
        except ValueError:
            continue
        age_days = max((now - published).total_seconds() / 86400, 0.25)
        views = int(stats.get("viewCount", 0))
        likes = int(stats.get("likeCount", 0))
        results.append({
            "platform": "YouTube", "topic": found.get(video_id, {}).get("topic", ""),
            "title": snippet.get("title", "Untitled"),
            "creator": snippet.get("channelTitle", ""),
            "url": f"https://www.youtube.com/watch?v={video_id}",
            "views": views, "likes": likes,
            "estimated_views_per_day": round(views / age_days),
            "duration_seconds": seconds,
            "published_at": snippet.get("publishedAt", ""),
            "download_status": "Link only: YouTube audiovisual download not enabled",
        })
    results.sort(key=lambda x: x["estimated_views_per_day"], reverse=True)
    return results[:env_int("TOP_YOUTUBE", 10, 1, 30)]


def choose_file(video: dict[str, Any]) -> dict[str, Any] | None:
    files = [f for f in video.get("video_files", []) if f.get("file_type") == "video/mp4" and f.get("link") and ".m3u8" not in f["link"]]
    # Prefer portrait Full HD for shorts, else highest available up to 1080p.
    portrait = [f for f in files if (f.get("height") or 0) > (f.get("width") or 0)]
    pool = portrait or files
    if not pool:
        return None
    return min(pool, key=lambda f: (
        0 if (f.get("height") or 0) <= 1920 else 1,
        abs((f.get("width") or 0) - 1080),
        abs((f.get("height") or 0) - 1920),
    ))


def discover_pexels(api_key: str) -> list[dict[str, Any]]:
    if not api_key:
        print("[Pexels] API key missing: no licensed downloads available.")
        return []
    mode = os.getenv("PEXELS_MODE", "search").lower().strip()
    query = os.getenv("PEXELS_QUERY", "gaming").strip()
    is_popular = mode == "popular" or not query
    url = f"{PEXELS_API}/popular" if is_popular else f"{PEXELS_API}/search"
    params: dict[str, Any] = {"per_page": 20, "page": 1}
    if not is_popular:
        params.update({"query": query, "orientation": os.getenv("PEXELS_ORIENTATION", "portrait")})
    try:
        response = api_get(url, params=params, headers={"Authorization": api_key})
    except (requests.RequestException, ValueError) as e:
        print(f"[Pexels] API error: {safe_error(e)}")
        return []
    videos = []
    for v in response.get("videos", []):
        best = choose_file(v)
        if best is None:
            continue
        user = v.get("user") or {}
        videos.append({
            "platform": "Pexels", "id": str(v.get("id", "")),
            "title": f"{query or 'Popular'} — clip {v.get('id', '')}",
            "creator": user.get("name", "Pexels contributor"),
            "url": v.get("url", "https://www.pexels.com"),
            "duration_seconds": v.get("duration", 0),
            "license_url": "https://www.pexels.com/license/",
            "download_url": best["link"],
            "video_width": best.get("width"),
            "video_height": best.get("height"),
            "download_status": "Pending",
        })
    # API's popular ordering when mode=popular; search results retain provider order.
    return videos


def download_video(url: str, path: Path, max_size_bytes: int) -> bool:
    parsed = urlparse(url)
    # Accept only video file links returned by official Pexels API, on its CDN.
    if parsed.scheme != "https" or parsed.hostname not in {"videos.pexels.com", "www.pexels.com", "player.vimeo.com"}:
        print(f"[Pexels] Skipped unexpected file host: {parsed.hostname}")
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(".part")
    try:
        with requests.get(url, stream=True, timeout=(10, 120), headers={"User-Agent": USER_AGENT}) as r:
            r.raise_for_status()
            mime = r.headers.get("Content-Type", "").lower()
            if not ("video/" in mime or "octet-stream" in mime):
                print(f"[Pexels] Rejected non-video MIME: {mime}")
                return False
            length = int(r.headers.get("Content-Length") or 0)
            if length > max_size_bytes:
                print(f"[Pexels] Skipping over-size clip: {length / 1e6:.1f} MB")
                return False
            size = 0
            with tmp_path.open("wb") as f:
                for chunk in r.iter_content(chunk_size=256 * 1024):
                    if not chunk:
                        continue
                    size += len(chunk)
                    if size > max_size_bytes:
                        print("[Pexels] Download exceeded size limit: skipped.")
                        return False
                    f.write(chunk)
        if tmp_path.stat().st_size == 0:
            return False
        tmp_path.replace(path)
        return True
    except (requests.RequestException, OSError, ValueError) as e:
        print(f"[Pexels] Download error: {safe_error(e)}")
        return False
    finally:
        if tmp_path.exists():
            tmp_path.unlink()


def telegram_request(token: str, method: str, *, data: dict[str, Any] | None = None, files=None) -> dict[str, Any]:
    response = requests.post(f"{TG_API}/bot{token}/{method}", data=data, files=files, timeout=(10, 120))
    response.raise_for_status()
    payload = response.json()
    if not payload.get("ok"):
        raise RuntimeError(f"Telegram API error: {payload.get('description', 'unknown')}")
    return payload


def get_chat_id(token: str) -> None:
    if not token:
        print("Set TELEGRAM_BOT_TOKEN in .env first.")
        return
    try:
        result = api_get(f"{TG_API}/bot{token}/getUpdates")
        ids = sorted({str(m["message"]["chat"]["id"]) for m in result.get("result", []) if m.get("message", {}).get("chat", {}).get("id")})
        if ids:
            print("Chat ID(s):", ", ".join(ids))
        else:
            print("Open your bot on Telegram, send /start, then run this command again.")
    except (requests.RequestException, ValueError) as e:
        print(f"Telegram error: {safe_error(e)}")


def send_telegram(token: str, chat_id: str, youtube: list[dict[str, Any]], downloads: list[dict[str, Any]]) -> None:
    if not token or not chat_id:
        print("[Telegram] Not configured, files are saved locally.")
        return
    lines = ["🎬 Viral Video Finder — daily digest", "", f"YouTube links: {len(youtube)} | Downloaded licensed clips: {len(downloads)}", ""]
    for i, v in enumerate(youtube[:8], 1):
        title = v["title"][:85]
        lines.append(f"{i}. {title}\n{v['views']:,} views | ~{v['estimated_views_per_day']:,} views/day\n{v['url']}")
    lines += ["", "Source of downloadable clips: Pexels (license applies)", "https://www.pexels.com/license/"]
    msg = "\n\n".join(lines)
    try:
        telegram_request(token, "sendMessage", data={"chat_id": chat_id, "text": msg[:4000], "disable_web_page_preview": "true"})
    except (requests.RequestException, RuntimeError, ValueError) as e:
        print(f"[Telegram] Message failed: {safe_error(e)}")
    if os.getenv("TELEGRAM_SEND_VIDEOS", "true").lower() != "true":
        return
    for i, v in enumerate(downloads, 1):
        path = Path(v["saved_path"])
        try:
            with path.open("rb") as f:
                telegram_request(token, "sendVideo", data={
                    "chat_id": chat_id,
                    "caption": f"Clip {i} | {v['creator']}\n{v['url']}\nLicense: https://www.pexels.com/license/"[:1000],
                    "supports_streaming": "true",
                }, files={"video": (path.name, f, "video/mp4")})
            print(f"[Telegram] Sent {path.name}")
        except (requests.RequestException, RuntimeError, OSError) as e:
            print(f"[Telegram] Could not send {path.name}: {safe_error(e)}")
        time.sleep(1)


def write_report(youtube: list[dict[str, Any]], pexels: list[dict[str, Any]]) -> None:
    esc = lambda x: html.escape(str(x), quote=True)
    OUTPUT.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).isoformat()
    (OUTPUT / "report.json").write_text(json.dumps({
        "generated_at_utc": stamp, "youtube": youtube, "pexels": pexels,
        "disclaimer": "YouTube links only; downloadable files are from Pexels under its license.",
    }, indent=2, ensure_ascii=False), encoding="utf-8")

    def row(v: dict[str, Any], kind: str) -> str:
        esc = lambda x: html.escape(str(x), quote=True)
        url = esc(v.get("url", ""))
        if kind == "youtube":
            meta = f"{v.get('views', 0):,} views · ~{v.get('estimated_views_per_day', 0):,} views/day · {v.get('duration_seconds', 0)}s"
            badge = "Discovery only — no video download"
            local = ""
        else:
            meta = f"Contributor: {v.get('creator', '')} · {v.get('duration_seconds', 0)}s"
            badge = v.get("download_status", "Not downloaded")
            local = f" · Saved: {esc(v['saved_path'])}" if v.get("saved_path") else ""
        return f'<article><a href="{url}" target="_blank" rel="noopener noreferrer">{esc(v.get("title", "Video"))}</a><p>{esc(meta)}<br><small>{esc(badge)}{local}</small></p></article>'

    yt_cards = "".join(row(v, "youtube") for v in youtube) or "<p>No YouTube results.</p>"
    px_cards = "".join(row(v, "pexels") for v in pexels) or "<p>No Pexels results.</p>"
    page = f'''<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Viral Video Finder Report</title>
<style>body{{font:16px system-ui,sans-serif;background:#0b1220;color:#e7eefc;max-width:960px;margin:30px auto;padding:16px}}.columns{{display:grid;grid-template-columns:repeat(auto-fit,minmax(300px,1fr));gap:20px}}section{{min-width:0}}article{{border:1px solid #31405a;background:#162238;padding:15px;margin:12px 0;border-radius:12px;overflow-wrap:anywhere}}a{{color:#81c4ff;text-decoration:none;font-weight:600}}p,small{{color:#b6c3d7;line-height:1.5}}.note{{padding:14px;border:1px solid #506584;border-radius:10px}}</style></head><body><h1>🎬 Viral Video Finder</h1><p>Generated {esc(stamp)} (UTC)</p><p class="note">YouTube: links and discovery only. Downloaded MP4 files: Pexels, subject to <a href="https://www.pexels.com/license/">Pexels License</a>. Source: <a href="https://www.pexels.com">Pexels</a>.</p><div class="columns"><section><h2>YouTube popular recent videos ({len(youtube)})</h2>{yt_cards}</section><section><h2>Pexels licensed clips ({len(pexels)})</h2>{px_cards}</section></div></body></html>'''
    (OUTPUT / "report.html").write_text(page, encoding="utf-8")


def run(dry_run: bool) -> None:
    OUTPUT.mkdir(parents=True, exist_ok=True)
    yt = discover_youtube(os.getenv("YOUTUBE_API_KEY", ""))
    px = discover_pexels(os.getenv("PEXELS_API_KEY", ""))
    max_downloads = env_int("MAX_DOWNLOADS", 3, 1, 10)
    max_size_bytes = env_int("MAX_FILE_MB", 45, 1, 49) * 1024 * 1024
    downloaded: list[dict[str, Any]] = []
    if not dry_run:
        for v in px:
            if len(downloaded) >= max_downloads:
                break
            filename = f"pexels_{v['id']}.mp4"
            path = DOWNLOADS / filename
            if path.exists():
                print(f"[Pexels] Already present: {filename}")
                ok = True
            else:
                ok = download_video(v["download_url"], path, max_size_bytes)
            if ok:
                v["download_status"] = "Downloaded under Pexels License"
                v["saved_path"] = str(path)
                downloaded.append(v)
            else:
                v["download_status"] = "Not downloaded (size/network/type/host restriction)"
    else:
        for v in px:
            v["download_status"] = "Dry run: not downloaded"
    # Avoid persisting temporary direct CDN access URLs to public reports.
    for v in px:
        v.pop("download_url", None)
    write_report(yt, px)
    if not dry_run:
        send_telegram(os.getenv("TELEGRAM_BOT_TOKEN", ""), os.getenv("TELEGRAM_CHAT_ID", ""), yt, downloaded)
    print(f"Complete: {len(yt)} YouTube links; {len(downloaded)} licensed MP4 downloads.")
    print(f"Open {OUTPUT / 'report.html'} or inspect {OUTPUT / 'report.json'}")
    if not yt and not px:
        print("No results. Add a valid YOUTUBE_API_KEY and/or PEXELS_API_KEY to .env (or environment secrets).")


if __name__ == "__main__":
    cli = argparse.ArgumentParser(description="Find viral video links, download licensed Pexels clips")
    cli.add_argument("--dry-run", action="store_true", help="Discover only, do not download or send Telegram messages")
    cli.add_argument("--get-chat-id", action="store_true", help="Print your Telegram bot's recent chat IDs")
    args = cli.parse_args()
    if args.get_chat_id:
        get_chat_id(os.getenv("TELEGRAM_BOT_TOKEN", ""))
    else:
        run(args.dry_run)


In [ ]:
import sys, subprocess, zipfile
from pathlib import Path
from google.colab import files

# Run agent. It writes report.json, report.html and, where available, licensed MP4s.
result = subprocess.run([sys.executable, 'agent.py'], text=True, check=False)
if result.returncode:
    raise RuntimeError(f'Agent returned error code {result.returncode}; check the logs above.')

out = Path('output')
archive = Path('viral_video_results.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for item in out.rglob('*'):
        if item.is_file():
            z.write(item, item.relative_to(out.parent))
print(f'✅ Downloading {archive.name} to your phone ...')
files.download(str(archive))
